# 00 — Data Preparation
### Shared preprocessing pipeline used by all models
Run this notebook FIRST before any model notebook.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports complete.')

In [ ]:
# ── FILE PATHS ───────────────────────────────────────────────────────
HEALTHY_FILE   = 'MBP_ControllerData_0521760_Overlock.xlsx'
SYNTHETIC_FILE = 'Synthetic_Breakdown_Dataset_Final.xlsx'
REAL_BD_FILE   = 'Real_Breakdown_Records_-_Overlock.xlsx'

# ── SETTINGS ─────────────────────────────────────────────────────────
TIME_STEPS = 7

KNOWN_BREAKDOWNS = [
    'Waveness',
    'High Foot Pressure',
    'Skip Stitches/Slip',
    'Blade Blunt',
]
ALLOWED_STATES = ['Healthy'] + KNOWN_BREAKDOWNS

# ── FREQUENCY BAND GROUPS (for FAG-TFT) ─────────────────────────────
# Physically motivated grouping of vibration frequency bands
VIB_GROUP_1 = [f'{i}-{i+10}Hz' for i in range(10,  100, 10)]   # 10-100Hz  mechanical
VIB_GROUP_2 = [f'{i}-{i+10}Hz' for i in range(100, 300, 10)]   # 100-300Hz bearing
VIB_GROUP_3 = [f'{i}-{i+10}Hz' for i in range(300, 610, 10)]   # 300-610Hz blade/cutting
ELEC_FEATS  = ['machineVoltageMean','machineVoltageMin','machineVoltageMax',
               'machineCurrentMean','machineCurrentMin','machineCurrentMax']

print(f'✅ Config loaded.')
print(f'   VIB Group 1 (10-100Hz)  : {len(VIB_GROUP_1)} bands')
print(f'   VIB Group 2 (100-300Hz) : {len(VIB_GROUP_2)} bands')
print(f'   VIB Group 3 (300-610Hz) : {len(VIB_GROUP_3)} bands')
print(f'   Electrical features     : {len(ELEC_FEATS)}')

In [ ]:
# ── LOAD DATA ────────────────────────────────────────────────────────
df_healthy   = pd.read_excel(HEALTHY_FILE)
df_synthetic = pd.read_excel(SYNTHETIC_FILE)
df_real_bd   = pd.read_excel(REAL_BD_FILE)

def filter_valid(df):
    return df[df['machineVibration'].astype(str).str.startswith('10')].copy()

df_healthy   = filter_valid(df_healthy)
df_synthetic = filter_valid(df_synthetic)
df_real_bd   = filter_valid(df_real_bd)

df_healthy['Breakdown'] = df_healthy['Breakdown'].fillna('Healthy')
df_healthy   = df_healthy[df_healthy['Breakdown'] == 'Healthy']
df_breakdown = pd.concat([df_synthetic, df_real_bd], ignore_index=True)
df_breakdown = df_breakdown[df_breakdown['Breakdown'].isin(KNOWN_BREAKDOWNS)]
master_df    = pd.concat([df_healthy, df_breakdown], ignore_index=True)
master_df    = master_df[master_df['Breakdown'].isin(ALLOWED_STATES)]

print(f'✅ Data loaded.')
print(f'   Healthy    : {len(df_healthy)}')
print(f'   Breakdown  : {len(df_breakdown)}')
print(f'   Total      : {len(master_df)}')
print(master_df['Breakdown'].value_counts())

In [ ]:
# ── FEATURE EXTRACTION ───────────────────────────────────────────────
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)

    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df  = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    return pd.concat([vib_df.reset_index(drop=True), elec_df], axis=1)

X_all = extract_features(master_df)
y_all = master_df['Breakdown'].values
print(f'✅ Features extracted: {X_all.shape}  (samples x 66 features)')

In [ ]:
# ── ENCODE LABELS ────────────────────────────────────────────────────
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_all)
num_classes = len(encoder.classes_)
print(f'✅ Classes: {list(encoder.classes_)}')

# ── TRAIN/TEST SPLIT ─────────────────────────────────────────────────
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all.values, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# ── SCALE (fit on train only) ────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print(f'   Train: {X_train_scaled.shape}  Test: {X_test_scaled.shape}')

In [ ]:
# ── SEQUENCE CREATION ────────────────────────────────────────────────
def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, TIME_STEPS)
X_test_seq,  y_test_seq  = create_sequences(X_test_scaled,  y_test,  TIME_STEPS)

print(f'✅ Sequences created.')
print(f'   Train: {X_train_seq.shape}  →  (samples, {TIME_STEPS}, 66)')
print(f'   Test : {X_test_seq.shape}')

In [ ]:
# ── HEALTHY-ONLY DATA FOR AUTOENCODER ────────────────────────────────
healthy_mask = master_df['Breakdown'].values == 'Healthy'
X_healthy    = extract_features(master_df[healthy_mask])
X_healthy_scaled = scaler.transform(X_healthy.values)

def create_sequences_X(X, time_steps):
    return np.array([X[i:i+time_steps] for i in range(len(X)-time_steps)])

X_healthy_seq = create_sequences_X(X_healthy_scaled, TIME_STEPS)
split = int(len(X_healthy_seq)*0.8)
X_ae_train, X_ae_test = X_healthy_seq[:split], X_healthy_seq[split:]

print(f'✅ Autoencoder sequences: train={X_ae_train.shape}  test={X_ae_test.shape}')

In [ ]:
# ── SAVE ALL PREPARED DATA ───────────────────────────────────────────
prepared = {
    'X_train_seq'  : X_train_seq,
    'X_test_seq'   : X_test_seq,
    'y_train_seq'  : y_train_seq,
    'y_test_seq'   : y_test_seq,
    'X_ae_train'   : X_ae_train,
    'X_ae_test'    : X_ae_test,
    'num_classes'  : num_classes,
    'num_features' : X_train_seq.shape[2],
    'TIME_STEPS'   : TIME_STEPS,
    'VIB_GROUP_1'  : VIB_GROUP_1,
    'VIB_GROUP_2'  : VIB_GROUP_2,
    'VIB_GROUP_3'  : VIB_GROUP_3,
    'ELEC_FEATS'   : ELEC_FEATS,
    'feature_names': list(X_all.columns),
}
with open('prepared_data.pkl','wb') as f: pickle.dump(prepared, f)
with open('scaler.pkl','wb') as f: pickle.dump(scaler, f)
with open('encoder.pkl','wb') as f: pickle.dump(encoder, f)

print('✅ Saved: prepared_data.pkl  scaler.pkl  encoder.pkl')
print(f'   num_classes  : {num_classes}')
print(f'   num_features : {X_train_seq.shape[2]}')
print(f'   TIME_STEPS   : {TIME_STEPS}')